In [3]:
import os
import numpy as np
import librosa
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from sklearn.metrics import accuracy_score, classification_report

In [4]:
non_music_path = "dataset1/non_music"
music_path = "dataset1/music"

In [5]:
non_music_files = [os.path.join(non_music_path, f) for f in os.listdir(non_music_path)]
music_files = [os.path.join(music_path, f) for f in os.listdir(music_path)]

In [6]:
def extract_features(file_path):
    audio, sr = librosa.load(file_path)
    # Extract features like MFCCs, chroma features, etc.
    # For simplicity, let's use only MFCCs for this example
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20)
    return mfccs.flatten()

In [8]:
non_music_features = np.array([extract_features(f) for f in non_music_files])

In [9]:

music_features = np.array([extract_features(f) for f in music_files])

In [10]:
non_music_labels = np.zeros(len(non_music_files))
music_labels = np.ones(len(music_files))

In [11]:
X = np.concatenate((non_music_features, music_features))
y = np.concatenate((non_music_labels, music_labels))

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [14]:
model = Sequential([
    Flatten(input_shape=(X_train.shape[1],)),
    Dense(2048, activation='relu'),
    Dense(1024, activation='relu'),
    Dense(512, activation='relu'),
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

c:\Users\Amine\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [15]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [22]:
model.fit(X_train, y_train, epochs=10, batch_size=1024, validation_split=0.2)


Epoch 1/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 45s 402ms/step - accuracy: 0.9980 - loss: 0.0047 - val_accuracy: 0.9855 - val_loss: 0.0734
Epoch 2/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 44s 389ms/step - accuracy: 0.9985 - loss: 0.0029 - val_accuracy: 0.9846 - val_loss: 0.0877
Epoch 3/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 43s 383ms/step - accuracy: 0.9987 - loss: 0.0025 - val_accuracy: 0.9853 - val_loss: 0.0945
Epoch 4/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 43s 382ms/step - accuracy: 0.9989 - loss: 0.0019 - val_accuracy: 0.9858 - val_loss: 0.0992
Epoch 5/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 44s 388ms/step - accuracy: 0.9988 - loss: 0.0023 - val_accuracy: 0.9860 - val_loss: 0.0985
Epoch 6/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 43s 385ms/step - accuracy: 0.9987 - loss: 0.0023 - val_accuracy: 0.9857 - val_loss: 0.0962
Epoch 7/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 44s 394ms/step - accuracy: 0.9986 - loss: 0.0031 - val_accuracy: 0.9850 - val_loss: 0.1029
Epoch 8/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 44s 396ms/step - accuracy: 0.9989 - loss: 0

In [23]:
y_pred = (model.predict(X_test) > 0.9).astype("int32")
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

1119/1119 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step
Accuracy: 0.986415853760796


In [24]:
print("Classification Report:")
print(classification_report(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

         0.0       0.99      0.98      0.99     17923
         1.0       0.98      0.99      0.99     17854

    accuracy                           0.99     35777
   macro avg       0.99      0.99      0.99     35777
weighted avg       0.99      0.99      0.99     35777



In [25]:
model.save("CNNmodelForMp3_99%.h5")